In [1]:
import numpy as np
import pandas as pd

import qubx  # type: ignore
import pyarrow as pa

%qubxd

%load_ext autoreload
%autoreload 2

# - - - - - - - - - -
from qubx.data.containers import RawData, RawMultiData

_mk_batch = lambda t,n: pa.RecordBatch.from_arrays([np.arange(t, t+n), np.arange(10.0 * t, 10.0 * t + n)], ["time", "value"])


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


### Common

In [2]:
pa.__version__

'15.0.2'

1. Play with pa

In [13]:
r1 = pa.RecordBatch.from_pandas(pd.DataFrame({"time": [1, 2, 3], "value": [4, 5, 6]}))
r2 = pa.RecordBatch.from_pandas(pd.DataFrame({"time": [11, 12, 13], "value": [24, 25, 26]}))

In [16]:
b1 = _mk_batch(1, 500)
b2 = _mk_batch(501, 500)
combined = pa.concat_tables([pa.Table.from_batches([b1]), pa.Table.from_batches([b2])]).combine_chunks().to_batches()[0] 
combined

pyarrow.RecordBatch
time: int64
value: double
----
time: [1,2,3,4,5,6,7,8,9,10,...,991,992,993,994,995,996,997,998,999,1000]
value: [10,11,12,13,14,15,16,17,18,19,...,5500,5501,5502,5503,5504,5505,5506,5507,5508,5509]

### Working with Batches as some storage

In [232]:
batches = []
S = 5_000
for k in range(0, 10*S, S):
    batches.append(_mk_batch(k, S))

In [231]:
%%timeit
combined = pa.concat_tables([pa.Table.from_batches([b]) for b in batches]).combine_chunks().to_batches()[0] 

56.8 μs ± 162 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [230]:
len(combined)

50000

In [ ]:
# for r in zip(*combined.to_pydict().values()):
#     break

In [ ]:
# for r in combined.to_pylist():
#     break

In [ ]:
t_c = combined.column("time").to_numpy(zero_copy_only=True)
v_c = combined.column("value").to_numpy(zero_copy_only=True)

In [238]:
_cs = 0 
for t,v in zip(t_c, v_c):
    _cs += v / (t + 1)
_cs

406444.6489152973

### RawData2

In [227]:
from qubx.data.storage import IDataTransformer, Transformable
from qubx.core.basics import DataType
import pyarrow as pa
from typing import Any
from qubx.data.storages.utils import find_time_col_idx

In [228]:
class RawData(Transformable):
    _create_key = object()
    __slots__ = ("data_id", "dtype", "raw", "_index")

    def __init__(self, data_id: str, dtype: DataType, record_batch: pa.RecordBatch, *, _key=None):
        if _key is not RawData._create_key:
            raise TypeError("Do not call RawData() directly, use RawData.from_*() methods")
        self.data_id = data_id
        self.dtype = dtype
        self.raw = record_batch
        self._index = find_time_col_idx(self.names)

    @property
    def names(self) -> list[str]:
        return list(self.raw.schema.names)

    @classmethod
    def from_record_batch(cls, data_id: str, dtype: DataType, data: pa.RecordBatch) -> "RawData":
        return cls(data_id, dtype, data, _key=cls._create_key)

    @classmethod
    def from_table(cls, data_id: str, dtype: DataType, data: pa.Table) -> "RawData":
        batches = data.combine_chunks().to_batches()
        batch = batches[0] if batches else pa.RecordBatch.from_pydict({n: [] for n in data.schema.names})
        return cls(data_id, dtype, batch, _key=cls._create_key)

    @classmethod
    def from_pandas(cls, data_id: str, dtype: DataType, data: pd.DataFrame | pd.Series) -> "RawData":
        if isinstance(data, pd.Series):
            _data = data.reset_index(name=data.name or "value")
            _data.columns = ["time", _data.columns[1]]
        else:
            _data = data.reset_index() if data.index.name else data
        return cls(data_id, dtype, pa.RecordBatch.from_pandas(_data), _key=cls._create_key)

    def __len__(self) -> int:
        return self.raw.num_rows

    def get_time_interval(self) -> tuple[int | None, int | None]:
        """
        Returns start and end timestamp from raw data
        """
        if len(self) == 0:
            return (None, None)
        _idx_col = self.raw.column(self._index)
        return (_idx_col[0].as_py(), _idx_col[-1].as_py())

    def transform(self, transformer: IDataTransformer) -> object:
        return transformer.process_data(self.data_id, self.dtype, self.raw, self.names, self._index)

    def __repr__(self) -> str:
        s, e = self.get_time_interval()
        _range = f"{s} : {e}" if s is not None else "EMPTY"
        return f"{self.data_id}({self.dtype})[{_range}] : ({len(self)} x {len(self.names)})"

In [229]:
RawData.from_record_batch("Test1", DataType.OHLC, combined)

Test1(ohlc)[0 : 49999] : (50000 x 2)

In [186]:
r1 = RawData.from_record_batch("x1", "ohlcv", _mk_batch(1, 500))
r2 = RawData.from_record_batch("x1", "ohlcv", _mk_batch(501, 1000))
print(r1)
print(r2)

print(
    RawData.from_pandas("X", "ohlc", pd.DataFrame({"date": [1, 2, 3], "value": [4, 5, 6]}).set_index("date")),
    "\n",
    RawData.from_pandas("X", "ohlc", pd.Series([1, 2, 3, 4], name="www"))
)

x1(ohlcv)[1 : 500] : (500 x 2)
x1(ohlcv)[501 : 1500] : (1000 x 2)
X(ohlc)[1 : 3] : (3 x 2) 
 X(ohlc)[0 : 3] : (4 x 2)


In [204]:
t1 = pa.Table.from_pandas(pd.DataFrame({"date": [0, 2, 3, 4, 5, 6], "value": [11,22,33,44,55,66]}))
x1 = RawData.from_table("x1", "ohlcv", t1[0:4]).raw.to_pandas()
x2 = RawData.from_table("x1", "ohlcv", t1[3:]).raw.to_pandas()

In [223]:
x1

,date,value
0,0,11
1,2,22
2,3,33
3,4,44


In [222]:
x2

,date,value
0,4,999
1,5,999
2,6,66


In [188]:
RawData.from_pandas("X", "ohlc", pd.DataFrame({"date": [], "value": []}))

X(ohlc)[EMPTY] : (0 x 2)

### RawData with nones

In [2]:
from qubx.data.transformers import OHLCVSeries, TypedRecords
from qubx.core.basics import DataType

In [3]:
raw_data = RawData.from_pandas("X", "ohlc", pd.DataFrame({
    "time":["2020-01-01", "2020-01-02", "2020-01-03"], 
    "open":  [1,2,3],
    "high":  [1,2,3],
    "low":   [3,2,1],
    "close": [1,2,None],
    "count": [None,None,None],
    "volume": [None,None,None],
}))

In [4]:
raw_data.to_pd()

,open,high,low,close,count,volume
time,,,,,,
2020-01-01,1,1,3,1.0,None,None
2020-01-02,2,2,2,2.0,None,None
2020-01-03,3,3,1,NaN,None,None


In [5]:
raw_data.to_ohlc()

            open  high  low  close  volume  bought_volume  volume_quote  \
timestamp                                                                 
2020-01-01   1.0   1.0  3.0    1.0     NaN            0.0           0.0   
2020-01-02   2.0   2.0  2.0    2.0     NaN            0.0           0.0   
2020-01-03   3.0   3.0  1.0    NaN     NaN            0.0           0.0   

            bought_volume_quote  trade_count  
timestamp                                     
2020-01-01                  0.0          0.0  
2020-01-02                  0.0          0.0  
2020-01-03                  0.0          0.0  

In [51]:

# ok, let's review build_snapshots in /home/quant0/devs/Qubx/src/qubx/data/storages/utils.py                                       
#   it's based on list[tuple] could we use columns from record banches instead ?  

In [6]:
raw_data.to_records()[0].bought_volume

0.0

In [12]:
raw_data.to_ticks(daily_session_start_end="STOCKS")

[[2020-01-01T09:30:01.100000000]	1.00000 (1000000000.00) | 1.00000 (1000000000.00),
 [2020-01-01T11:00:00.000000000]	3.00000 (1000000000.00) | 3.00000 (1000000000.00),
 [2020-01-01T13:00:00.000000000]	1.00000 (1000000000.00) | 1.00000 (1000000000.00),
 [2020-01-01T15:59:58.900000000]	1.00000 (1000000000.00) | 1.00000 (1000000000.00),
 [2020-01-02T09:30:01.100000000]	2.00000 (1000000000.00) | 2.00000 (1000000000.00),
 [2020-01-02T11:00:00.000000000]	2.00000 (1000000000.00) | 2.00000 (1000000000.00),
 [2020-01-02T13:00:00.000000000]	2.00000 (1000000000.00) | 2.00000 (1000000000.00),
 [2020-01-02T15:59:58.900000000]	2.00000 (1000000000.00) | 2.00000 (1000000000.00),
 [2020-01-03T09:30:01.100000000]	3.00000 (1000000000.00) | 3.00000 (1000000000.00),
 [2020-01-03T11:00:00.000000000]	3.00000 (1000000000.00) | 3.00000 (1000000000.00),
 [2020-01-03T13:00:00.000000000]	1.00000 (1000000000.00) | 1.00000 (1000000000.00),
 [2020-01-03T15:59:58.900000000]	nan (1000000000.00) | nan (1000000000.00)]

### RawMultiData

In [13]:
from qubx.data.containers import RawData, RawMultiData
from qubx.core.basics import DataType

In [14]:
d1 = RawData.from_pandas("X", "ohlc", pd.DataFrame({
        "y": [-1,-2,-3], "date": [1,2,3], "value": [11,22,33]
    }))
d2 = RawData.from_pandas("X2", "ohlc", pd.DataFrame({
        "y": [-1,-2,-3,0], "date": [12,22,32,55], "value": [11,22,33,99]
    }))

In [15]:
r1 = RawData.from_pandas(
    "TEST1",
    DataType.TRADE,
    pd.DataFrame({
        "price": [100,120,190], "time": [1000,2000,3000], "size": [0.5, 0.3, 1.5]
    })
)

In [16]:
dd = r1.data

In [17]:
dd.column_names

['price', 'time', 'size']

In [18]:
dm = RawMultiData([d1, d2])

In [22]:
dm.to_pd()

X         X2      
                                 y value    y value
date                                               
1970-01-01 00:00:00.000000001 -1.0  11.0  NaN   NaN
1970-01-01 00:00:00.000000002 -2.0  22.0  NaN   NaN
1970-01-01 00:00:00.000000003 -3.0  33.0  NaN   NaN
1970-01-01 00:00:00.000000012  NaN   NaN -1.0  11.0
1970-01-01 00:00:00.000000022  NaN   NaN -2.0  22.0
1970-01-01 00:00:00.000000032  NaN   NaN -3.0  33.0
1970-01-01 00:00:00.000000055  NaN   NaN  0.0  99.0

In [23]:
from qubx.data.registry import StorageRegistry

In [24]:
r = StorageRegistry.get("qdb::quantlab")["BINANCE.UM", "SWAP"]

In [34]:
r.read("BTCUSDT", "ohlc(1h)", "-5d", "now").to_ticks()

[[2026-01-28T16:00:01.000000000]	89655.80000 (1000000000.00) | 89655.80000 (1000000000.00),
 [2026-01-28T16:24:00.000000000]	89718.90000 (1000000000.00) | 89718.90000 (1000000000.00),
 [2026-01-28T16:36:00.000000000]	89419.40000 (1000000000.00) | 89419.40000 (1000000000.00),
 [2026-01-28T16:59:59.000000000]	89588.80000 (1000000000.00) | 89588.80000 (1000000000.00),
 [2026-01-28T17:00:01.000000000]	89588.80000 (1000000000.00) | 89588.80000 (1000000000.00),
 [2026-01-28T17:24:00.000000000]	89238.80000 (1000000000.00) | 89238.80000 (1000000000.00),
 [2026-01-28T17:36:00.000000000]	90592.70000 (1000000000.00) | 90592.70000 (1000000000.00),
 [2026-01-28T17:59:59.000000000]	90142.90000 (1000000000.00) | 90142.90000 (1000000000.00),
 [2026-01-28T18:00:01.000000000]	90142.90000 (1000000000.00) | 90142.90000 (1000000000.00),
 [2026-01-28T18:24:00.000000000]	90149.20000 (1000000000.00) | 90149.20000 (1000000000.00),
 [2026-01-28T18:36:00.000000000]	89216.40000 (1000000000.00) | 89216.40000 (1000